# Persistence & Checkpointing：InMemorySaver

> 适用版本：本项目锁定的 **LangGraph 1.1.2** 与 **langgraph-checkpoint 4.0.1**。本笔记不调用大模型或外部 API。

LangGraph 的 persistence（持久化）层会在图执行过程中保存 checkpoint（检查点）。一个 checkpoint 是某个时刻的 `StateSnapshot`：它包含当时的 State 值、下一批待执行节点、元数据，以及前一个 checkpoint 的引用。多个 checkpoint 通过 `thread_id` 组织成一条线程时间线。

| 概念 | 作用 | 是否属于业务 State |
| --- | --- | --- |
| State | 节点读取和更新的业务数据 | 是 |
| checkpoint | 某个 superstep 边界上的 State 快照与调度信息 | 否 |
| `thread_id` | 定位一条 checkpoint 时间线；相同 ID 恢复同一线程，不同 ID 相互隔离 | 否 |
| checkpointer | 负责 checkpoint 与节点级 pending writes 的读写 | 否 |

`thread_id` 放在 `config["configurable"]` 中，而不是 State 中：

```python
config = {"configurable": {"thread_id": "counter-demo"}}
```

本笔记将验证：

1. 首次 `invoke` 如何创建 checkpoint；
2. 第二次使用相同 `thread_id` 时，State 如何从上一次结果继续累积；
3. 不同 `thread_id` 为什么不会共享 State；
4. `InMemorySaver` 的生命周期、适用场景与限制；
5. 为什么历史快照里的 `task.result` 看起来出现在结果合并前；
6. 第二次 `invoke` 是否重写旧 checkpoint，以及三种 durability 是否改变最终 `values`。

```mermaid
flowchart LR
    I1[第一次 invoke: delta=5] --> C1[configurable.thread_id=memory-counter-demo]
    C1 --> L1[读取该线程最新 checkpoint: 首次为空]
    L1 --> A1[accumulate: total=5]
    A1 --> P1[保存 superstep checkpoint]
    P1 --> S1[summarize: 生成摘要]
    S1 --> P2[保存完成 checkpoint]
    P2 --> I2[第二次 invoke: delta=3 且 thread_id 相同]
    I2 --> L2[恢复 total=5 与历史记录]
    L2 --> A2[accumulate: total=8]
    A2 --> P3[追加新的 checkpoints]
```


In [12]:
import operator
from importlib.metadata import version
from pprint import pprint
from typing import Annotated, TypedDict

from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph

print("LangGraph version:", version("langgraph"))
print("Checkpoint version:", version("langgraph-checkpoint"))


LangGraph version: 1.1.2
Checkpoint version: 4.0.1


## 1. 创建一个不依赖模型的状态图

下面使用 `accumulate → summarize` 两个顺序节点：

- 每次调用只输入本轮增量 `delta`；
- `accumulate` 读取 checkpoint 恢复出来的 `total`，再加上本轮 `delta`；
- `history` 使用 `operator.add` reducer，因此节点返回的新列表会追加到历史，而不是覆盖旧列表；
- `summarize` 根据最新总数生成文本摘要。

`TypedDict(total=False)` 让所有字段在类型层面都可省略。首次调用还没有 `total`，节点用 `state.get("total", 0)` 提供初始值；后续调用则从 checkpoint 中读到已有值。


In [2]:
class CounterState(TypedDict, total=False):
    delta: int
    total: int
    history: Annotated[list[str], operator.add]
    summary: str


def accumulate(state: CounterState) -> dict[str, int | list[str]]:
    """把本轮 delta 累加到 checkpoint 恢复出的 total。"""

    previous_total = state.get("total", 0)
    delta = state["delta"]
    new_total = previous_total + delta
    return {
        "total": new_total,
        "history": [f"{previous_total} + {delta} = {new_total}"],
    }


def summarize(state: CounterState) -> dict[str, str]:
    """根据已经提交的 total 生成本轮摘要。"""

    return {"summary": f"当前累计值：{state['total']}"}


def build_counter_graph(checkpointer):
    """构建同一张教学图；切换 saver 时只替换持久化后端。"""

    builder = StateGraph(state_schema=CounterState)
    builder.add_node("accumulate", accumulate)
    builder.add_node("summarize", summarize)
    builder.add_edge(START, "accumulate")
    builder.add_edge("accumulate", "summarize")
    builder.add_edge("summarize", END)
    return builder.compile(checkpointer=checkpointer)


memory_saver = InMemorySaver()
memory_graph = build_counter_graph(memory_saver)

print("图已使用 InMemorySaver 编译。")


图已使用 InMemorySaver 编译。


## 2. 第一次调用：创建线程与 checkpoint

`compile(checkpointer=...)` 只是为图装配 saver；真正运行时还必须提供 `thread_id`。这里先删除同名教学线程，让该单元格重复执行时仍从干净状态开始。`delete_thread()` 只清理这个明确指定的 ID，不影响其他线程。

本教程显式使用 `durability="sync"`：每个 superstep 的 checkpoint 会在下一步开始前同步写入 saver，便于紧接着检查历史。


In [3]:
MEMORY_THREAD_ID = "memory-counter-demo"
thread_config: RunnableConfig = {
    "configurable": {"thread_id": MEMORY_THREAD_ID}
}

# 仅清理本教程使用的确定性线程，保证从头运行 Notebook 时输出稳定。
memory_saver.delete_thread(MEMORY_THREAD_ID)

first_result = memory_graph.invoke(
    {"delta": 5},
    thread_config,
    durability="sync",
)

print("第一次调用结果：", first_result)

assert first_result["total"] == 5
assert first_result["history"] == ["0 + 5 = 5"]
assert first_result["summary"] == "当前累计值：5"


第一次调用结果： {'delta': 5, 'total': 5, 'history': ['0 + 5 = 5'], 'summary': '当前累计值：5'}


### 结果解读

- 首次没有旧 checkpoint，因此 `total` 从节点提供的默认值 `0` 开始；
- 输入中的 `delta=5` 先写入线程 State，`accumulate` 再返回 `total=5` 和一条历史；
- `summarize` 在下一 superstep 读取已经提交的 `total=5`；
- 返回结果是当前最新 State，不是 checkpoint 底层序列化对象。


## 3. 检查 checkpoint 历史与保存时机

`get_state_history(config)` 返回**从新到旧**的 `StateSnapshot`。为了按执行顺序观察，下面先反转列表。对这张顺序图，LangGraph 1.1.2 的首次完整调用可看到四个边界：

| `metadata.step` | 代表的边界 | `next` |
| ---: | --- | --- |
| `-1` | 初始输入边界 | `__start__` |
| `0` | 本轮输入已写入 State | `accumulate` |
| `1` | `accumulate` 更新已提交 | `summarize` |
| `2` | `summarize` 更新已提交，图完成 | 空元组 |

> `step` 数字适合调试，不应作为业务主键。稳定定位某个历史快照应使用该快照 `configurable` 中的 `checkpoint_id`。


In [4]:
first_history_newest_first = list(
    memory_graph.get_state_history(thread_config)
)
first_history = list(reversed(first_history_newest_first))

for snapshot in first_history:
    print(
        f"step={snapshot.metadata['step']:>2} | "
        f"source={snapshot.metadata['source']:<5} | "
        f"next={snapshot.next} | values={dict(snapshot.values)}"
    )

checkpoint_ids = [
    snapshot.config["configurable"]["checkpoint_id"]
    for snapshot in first_history
]

assert [snapshot.metadata["step"] for snapshot in first_history] == [
    -1, 0, 1, 2
]
assert len(checkpoint_ids) == len(set(checkpoint_ids)) == 4
assert first_history[-1].next == ()
print("checkpoint 数量：", len(first_history))
print("每个 checkpoint 都有唯一 checkpoint_id：", True)


step=-1 | source=input | next=('__start__',) | values={'history': []}
step= 0 | source=loop  | next=('accumulate',) | values={'delta': 5, 'history': []}
step= 1 | source=loop  | next=('summarize',) | values={'delta': 5, 'total': 5, 'history': ['0 + 5 = 5']}
step= 2 | source=loop  | next=() | values={'delta': 5, 'total': 5, 'history': ['0 + 5 = 5'], 'summary': '当前累计值：5'}
checkpoint 数量： 4
每个 checkpoint 都有唯一 checkpoint_id： True


## 4. 相同 `thread_id` 的第二次调用：恢复旧 State 后再执行

第二次只输入 `delta=3`，没有再次传入 `total` 或 `history`。LangGraph 会先按 `thread_id` 读取最新 checkpoint，将新输入合并到恢复出的 State，然后从 `START` 发起本轮新的图执行。

> 这里是“已完成线程上的新一轮调用”，不是从 `END` 节点继续执行。若图因 `interrupt()` 暂停，则应在相同 `thread_id` 上使用 `Command(resume=...)` 恢复中断点，那是另一种恢复语义。


In [5]:
first_latest_checkpoint_id = first_history_newest_first[0].config[
    "configurable"
]["checkpoint_id"]

second_result = memory_graph.invoke(
    {"delta": 3},
    thread_config,
    durability="sync",
)
latest_snapshot = memory_graph.get_state(thread_config)
all_history = list(memory_graph.get_state_history(thread_config))
second_latest_checkpoint_id = latest_snapshot.config["configurable"][
    "checkpoint_id"
]

print("第二次调用结果：", second_result)
print("最新 checkpoint 的 next：", latest_snapshot.next)
print("两轮调用后的 checkpoint 总数：", len(all_history))

assert second_result["total"] == 8
assert second_result["history"] == [
    "0 + 5 = 5",
    "5 + 3 = 8",
]
assert latest_snapshot.values == second_result
assert latest_snapshot.next == ()
assert second_latest_checkpoint_id != first_latest_checkpoint_id
assert len(all_history) == 8
# thread_id 属于运行配置，不会混入节点读取的业务 State。
assert "thread_id" not in latest_snapshot.values


第二次调用结果： {'delta': 3, 'total': 8, 'history': ['0 + 5 = 5', '5 + 3 = 8'], 'summary': '当前累计值：8'}
最新 checkpoint 的 next： ()
两轮调用后的 checkpoint 总数： 8


In [6]:
for snapshot in all_history:
    print(
        f"step={snapshot.metadata['step']:>2} | "
        f"source={snapshot.metadata['source']:<5} | "
        f"next={snapshot.next} | values={dict(snapshot.values)}"
    )

step= 6 | source=loop  | next=() | values={'delta': 3, 'total': 8, 'history': ['0 + 5 = 5', '5 + 3 = 8'], 'summary': '当前累计值：8'}
step= 5 | source=loop  | next=('summarize',) | values={'delta': 3, 'total': 8, 'history': ['0 + 5 = 5', '5 + 3 = 8'], 'summary': '当前累计值：5'}
step= 4 | source=loop  | next=('accumulate',) | values={'delta': 3, 'total': 5, 'history': ['0 + 5 = 5'], 'summary': '当前累计值：5'}
step= 3 | source=input | next=('__start__',) | values={'delta': 5, 'total': 5, 'history': ['0 + 5 = 5'], 'summary': '当前累计值：5'}
step= 2 | source=loop  | next=() | values={'delta': 5, 'total': 5, 'history': ['0 + 5 = 5'], 'summary': '当前累计值：5'}
step= 1 | source=loop  | next=('summarize',) | values={'delta': 5, 'total': 5, 'history': ['0 + 5 = 5']}
step= 0 | source=loop  | next=('accumulate',) | values={'delta': 5, 'history': []}
step=-1 | source=input | next=('__start__',) | values={'history': []}


### 结果解读

- `total` 从第一次结束时的 `5` 恢复，再加 `3` 得到 `8`；
- `history` 原有列表来自 checkpoint，本轮节点返回的列表通过 reducer 追加；
- 两轮调用各产生 4 个 checkpoint，因此本例历史共有 8 个；
- 最新快照的 `next=()` 表示当前线程已完成，没有待调度节点；
- 新 checkpoint 通过 `parent_config` 串到旧 checkpoint，构成可查询的时间线。


## 5. 线程隔离：更换 `thread_id` 就从独立状态开始

checkpointer 的索引不只包含 `thread_id`，底层还会使用 `checkpoint_ns` 与 `checkpoint_id`。根图的常规调用通常只需要显式提供 `thread_id`；`checkpoint_ns` 主要用于子图命名空间，`checkpoint_id` 用于定位历史中的特定快照。

业务上应选择稳定且不会碰撞的线程标识，例如会话 ID、工单 ID 或工作流实例 ID。不要把多个用户误用同一个固定 ID。


In [7]:
other_thread_config: RunnableConfig = {
    "configurable": {"thread_id": "memory-counter-other"}
}
memory_saver.delete_thread("memory-counter-other")

other_thread_result = memory_graph.invoke(
    {"delta": 2},
    other_thread_config,
    durability="sync",
)
original_thread_snapshot = memory_graph.get_state(thread_config)

print("新线程结果：", other_thread_result)
print("原线程仍保持：", dict(original_thread_snapshot.values))

assert other_thread_result["total"] == 2
assert original_thread_snapshot.values["total"] == 8
assert len(list(memory_graph.get_state_history(other_thread_config))) == 4


新线程结果： {'delta': 2, 'total': 2, 'history': ['0 + 2 = 2'], 'summary': '当前累计值：2'}
原线程仍保持： {'delta': 3, 'total': 8, 'history': ['0 + 5 = 5', '5 + 3 = 8'], 'summary': '当前累计值：8'}


## 6. 边界验证：缺少 `thread_id` 与 saver 生命周期

下面验证两个常见误区：

1. 图已经配置 checkpointer，但调用时没给可定位线程的配置；
2. 使用相同 `thread_id`，却换成了一个全新的 `InMemorySaver` 对象。

第二种情况下 ID 虽然相同，新 saver 的内存里却没有旧 checkpoint，因此仍会从空状态开始。


In [8]:
try:
    memory_graph.invoke({"delta": 1}, durability="sync")
except ValueError as exc:
    missing_thread_error = str(exc)
    print("缺少 thread_id 的预期错误：", type(exc).__name__)
    print(missing_thread_error)
else:
    raise AssertionError("预期缺少 thread_id 时抛出 ValueError")

assert "thread_id" in missing_thread_error

fresh_saver = InMemorySaver()
fresh_graph = build_counter_graph(fresh_saver)
fresh_result = fresh_graph.invoke(
    {"delta": 4},
    thread_config,  # ID 相同，但 saver 已经不是原对象。
    durability="sync",
)

print("全新 InMemorySaver 上的结果：", fresh_result)
assert fresh_result["total"] == 4
assert memory_graph.get_state(thread_config).values["total"] == 8


缺少 thread_id 的预期错误： ValueError
Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id
全新 InMemorySaver 上的结果： {'delta': 4, 'total': 4, 'history': ['0 + 4 = 4'], 'summary': '当前累计值：4'}


## 7. 深入 checkpoint 时间线：`task.result` 为什么像“提前出现”

### 先给结论

- 正常完成时，`"sync"`、`"async"`、`"exit"` **不会改变节点计算语义或最终 `values`**；它们控制的是 checkpoint 写入何时等待、故障时最多可能缺少哪些持久化边界。
- 历史中某个 task 的 `result` 出现在“写入该结果之前”的 snapshot，并不是节点提前运行，也不是 durability 导致乱序。这个 snapshot 的 `values` 表示该 superstep **开始前已经提交的 State**，而 `next` / `tasks` 表示由它调度的任务。任务完成后，其 pending writes 仍归属于调度它的 checkpoint；稍后调用 `get_state_history()` 时，LangGraph 用这些 pending writes 重建 `tasks.result`。更新要到下一个 superstep 边界才合并进下一份 snapshot 的 `values`。
- `get_state_history()` 默认按**从新到旧**返回。若要按执行顺序阅读，必须先反转。

Graph API 会在 superstep 边界生成完整 checkpoint。同一 superstep 的并行节点读取相同的已提交 State，全部完成后才统一合并更新；已完成任务还会写 pending writes，以支持失败后的精确恢复。

`invoke` / `stream` 的 `durability` 控制持久化调度：

| 模式 | 保存与等待策略 | 正常完成后的计算结果 | 故障恢复边界 |
| --- | --- | --- | --- |
| `"sync"` | 每步的 checkpoint 在下一步开始前等待写完 | 不变 | 逐步持久化保证最强，写延迟进入关键路径 |
| `"async"`（默认） | checkpoint 写入与下一步计算并行 | 不变 | 突然退出时，最近一步可能仍在写入 |
| `"exit"` | 只在图退出时保存当时的当前 checkpoint | 不变 | 没有本次运行的中间 checkpoint 可恢复 |

> 本节用 `InMemorySaver` 验证逻辑不变量，不模拟进程崩溃，也不把内存写入耗时当作数据库落盘时序。`InMemorySaver` 本身不跨进程保留数据；生产恢复保证还取决于 PostgreSQL 等真实后端。


### 7.1 最小并行图与四个不同时间点

下面构造 `START → {left, right} → join → END`。

- `left` 与 `right` 属于同一个 superstep，并行读取相同的 `seed`；
- 二者都写 `branch_results`，因此该字段使用 `operator.add` reducer；
- `join` 只在两个分支都完成、结果已经合并后执行。

必须区分四件事：checkpoint 在边界创建、节点函数运行、任务写入 pending writes、下一边界合并 State。历史查询发生在运行结束之后，所以能把“后来产生的任务结果”展示在“当初调度该任务的 checkpoint”上。

```mermaid
flowchart LR
    C0["创建 step=0 checkpoint<br/>values: seed=10<br/>next: left, right"] --> R["left / right 节点运行<br/>各自产生 result"]
    R --> W["pending writes 归属 step=0<br/>按 task_id 保存"]
    W --> C1["创建 step=1 checkpoint<br/>合并 branch_results<br/>next: join"]
    C1 --> J["join 节点运行<br/>产生 joined"]
    J --> C2["创建 step=2 checkpoint<br/>最终 values"]
    H["运行结束后 get_state_history"] -.-> S["重建 step=0 snapshot<br/>values 仍是合并前<br/>tasks.result 已可见"]
```


In [10]:
class TimelineState(TypedDict, total=False):
    seed: int
    branch_results: Annotated[list[str], operator.add]
    joined: str


def left_branch(state: TimelineState) -> dict[str, list[str]]:
    """同一 superstep 的左分支；只返回自己的局部更新。"""

    return {"branch_results": [f"left:{state['seed'] + 1}"]}


def right_branch(state: TimelineState) -> dict[str, list[str]]:
    """同一 superstep 的右分支；只返回自己的局部更新。"""

    return {"branch_results": [f"right:{state['seed'] + 2}"]}


def join_branches(state: TimelineState) -> dict[str, str]:
    """在两个并行结果完成合并后生成稳定摘要。"""

    return {
        "joined": " | ".join(sorted(state["branch_results"]))
    }


def build_timeline_graph(checkpointer):
    builder = StateGraph(TimelineState)
    builder.add_node("left", left_branch)
    builder.add_node("right", right_branch)
    builder.add_node("join", join_branches)
    builder.add_edge(START, "left")
    builder.add_edge(START, "right")
    # 列表形式的起点表示 barrier：left 和 right 都完成后才运行 join。
    builder.add_edge(["left", "right"], "join")
    builder.add_edge("join", END)
    return builder.compile(checkpointer=checkpointer)


### 7.2 对照“创建时事件”与“事后历史”

先用 `stream_mode="checkpoints"` 记录 checkpoint 的**逻辑创建顺序**，再在图正常完成后调用 `get_state_history()`。二者使用同一组 `checkpoint_id`，但观察时点不同：

- checkpoint stream 事件展示边界刚创建时的调度视图；
- state history 会把后来写在该 checkpoint/task ID 下的 pending writes 重新组装为 `task.result` / `task.error`。

代码不预先拆出每一步变量，也不靠大量断言代替观察：它直接 `pprint` checkpoint stream 的原始事件，再完整展开 history 中每个 snapshot 的 `config/parent_config/metadata/values/next/tasks`。其中 task 会显示 `id/name/path/result/error/interrupts/state`。不要用列表下标猜 step，应该查看 `metadata.step` 与 `checkpoint_id`。


In [13]:
timeline_graph = build_timeline_graph(InMemorySaver())
timeline_config: RunnableConfig = {
    "configurable": {"thread_id": "task-result-timeline"}
}

print("checkpoint stream：按创建顺序直接打印原始事件\n")
for checkpoint_event in timeline_graph.stream(
        {"seed": 10},
        timeline_config,
        durability="sync",
        stream_mode="checkpoints",
):
    pprint(checkpoint_event, sort_dicts=False, width=100)
    print()

print("get_state_history() 原始返回顺序（新 → 旧）：")
print([
    snapshot.metadata["step"]
    for snapshot in timeline_graph.get_state_history(timeline_config)
])

print("\n反转后按 checkpoint 创建顺序打印完整 snapshot：")
for snapshot in reversed(
    list(timeline_graph.get_state_history(timeline_config))
):
    pprint(snapshot, width=100)
    print()

print("get_state() 返回的当前最新 snapshot：")
pprint(timeline_graph.get_state(timeline_config), width=100)


checkpoint stream：按创建顺序直接打印原始事件

{'config': {'configurable': {'checkpoint_ns': '',
                             'thread_id': 'task-result-timeline',
                             'checkpoint_id': '1f19b19e-9d78-6fac-bfff-fd2de794c1e6'}},
 'parent_config': None,
 'values': {'branch_results': []},
 'metadata': {'source': 'input', 'step': -1, 'parents': {}},
 'next': ['__start__'],
 'tasks': [{'id': '8dac3470-67e6-eb5e-b9e1-a0547faf9ffc',
            'name': '__start__',
            'interrupts': (),
            'state': None}]}

{'config': {'configurable': {'checkpoint_ns': '',
                             'thread_id': 'task-result-timeline',
                             'checkpoint_id': '1f19b19e-9d7e-6006-8000-0aa4b16741b9'}},
 'parent_config': {'configurable': {'checkpoint_ns': '',
                                    'thread_id': 'task-result-timeline',
                                    'checkpoint_id': '1f19b19e-9d78-6fac-bfff-fd2de794c1e6'}},
 'values': {'seed': 10, 'branch_results

### 结果解读：没有“结果穿越到过去”

以 `metadata.step=0` 为例：

1. 创建 checkpoint 时，`values.branch_results=[]`，并据此调度 `left` 与 `right`；
2. 两个节点随后运行，pending writes 记录在这个 checkpoint 下对应的 task ID；
3. 运行结束后查询历史时，`tasks` 会结合 pending writes 显示 `result`，但 checkpoint 自身已经提交的 `values` 不会被倒改；
4. 下一份 `metadata.step=1` checkpoint 才包含合并后的 `branch_results`。

因此，`tasks.result` 表示“由此 snapshot 调度的任务后来产生了什么”，不是“创建此 snapshot 之前已经存在什么”。`values` 才是该边界已经提交的 State。


### 7.3 用同一张图比较 `sync`、`async`、`exit`

每种模式都使用全新的 `InMemorySaver` 和独立 `thread_id`，避免历史相互污染。代码直接打印 `invoke()` 的原始返回值和每个原始 `StateSnapshot`；checkpoint/task ID 本来就会随机生成，观察时重点对照 `metadata.step`、`values`、`next` 与 task 的 `name/path/result/error`。

这里验证的是**正常完成后的不变量**，不通过强制杀进程伪造崩溃实验。真实故障中能恢复到哪个边界，还受故障发生时点与存储后端影响。


In [14]:
for durability_mode in ("sync", "async", "exit"):
    mode_graph = build_timeline_graph(InMemorySaver())
    mode_config: RunnableConfig = {
        "configurable": {
            "thread_id": f"durability-{durability_mode}"
        }
    }

    print(f"\n===== durability={durability_mode!r} =====")
    print("invoke() 原始返回值：")
    pprint(
        mode_graph.invoke(
            {"seed": 10},
            mode_config,
            durability=durability_mode,
        ),
        sort_dicts=False,
        width=100,
    )

    print("get_state_history() checkpoint 数量：", end=" ")
    print(len(list(mode_graph.get_state_history(mode_config))))
    print("按创建顺序打印原始 StateSnapshot：")
    for snapshot in reversed(
        list(mode_graph.get_state_history(mode_config))
    ):
        pprint(snapshot, width=100)
        print()



===== durability='sync' =====
invoke() 原始返回值：
{'seed': 10, 'branch_results': ['left:11', 'right:12'], 'joined': 'left:11 | right:12'}
get_state_history() checkpoint 数量： 4
按创建顺序打印原始 StateSnapshot：
StateSnapshot(values={'branch_results': []}, next=('__start__',), config={'configurable': {'thread_id': 'durability-sync', 'checkpoint_ns': '', 'checkpoint_id': '1f19b1c5-c9a5-6086-bfff-d19e3f979d7c'}}, metadata={'source': 'input', 'step': -1, 'parents': {}}, created_at='2026-08-18T15:49:14.020258+00:00', parent_config=None, tasks=(PregelTask(id='9760448a-4e53-a28f-ce47-1cb8c06d1e97', name='__start__', path=('__pregel_pull', '__start__'), error=None, interrupts=(), state=None, result={'seed': 10}),), interrupts=())

StateSnapshot(values={'seed': 10, 'branch_results': []}, next=('left', 'right'), config={'configurable': {'thread_id': 'durability-sync', 'checkpoint_ns': '', 'checkpoint_id': '1f19b1c5-c9a7-64ee-8000-d9ea34c744cb'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_

### durability 结论边界

- 从三段原始输出可以直接看到：三种模式的最终返回值相同；`sync` 与 `async` 在正常返回后拥有相同的 checkpoint/task 逻辑语义。
- `task.result` 出现在调度它的 snapshot，是 pending writes 与历史重建的语义，和 `sync` / `async` 的选择无关。
- `exit` 并不会改变计算结果，但它根本不保存本次运行的中间 checkpoint，所以不能从 history 观察中间 `tasks.result`。
- 本 Notebook 没有强制终止解释器，因此不声称实际测出了某个后端在崩溃瞬间丢失了哪一步。`sync`、`async` 的差异是持久化等待策略和故障窗口，不是 State 更新顺序。


### 7.4 同一 `thread_id` 的第二次 `invoke` 会重写旧 snapshot 吗？

**不会。** 第二次调用先按 `thread_id` 读取上一轮最新 checkpoint，再追加一条新的 checkpoint 链：

1. 读取上一轮最终 checkpoint；这是 read，不增加 checkpoint 数量；
2. 创建新的 input checkpoint：`values` 仍是恢复出的旧 State，`next=("__start__",)`；本次输入完成后会作为 `__start__` task 的 `result` 出现在这份 snapshot；
3. 创建下一 checkpoint，把本次输入合并进 State，再调度业务节点；
4. 后续 superstep 继续追加 checkpoint，旧 ID 与旧内容保持不变。

```mermaid
flowchart LR
    O["上一轮最新 step=2<br/>checkpoint_id=old<br/>values: seed=10"] --> R["get_tuple(thread_id)<br/>读取 old，不写入"]
    R --> I["新增 step=3 input checkpoint<br/>parent=old<br/>values 仍是旧 State<br/>next: __start__"]
    I --> M["新增 step=4 checkpoint<br/>合并本次 seed=20<br/>next: left, right"]
    M --> B["新增 step=5 checkpoint<br/>合并并行结果<br/>next: join"]
    B --> F["新增 step=6 checkpoint<br/>最终 values"]
```

下面直接使用普通 `InMemorySaver`：第一次调用后打印 `get_state()` 与完整 history，第二次调用后再次打印。通过旧 checkpoint ID 是否仍是 history 前缀、第一份新增 snapshot 的 `parent_config` 是否指向旧最新 ID，可以直接判断它是读取后追加，而不是原样重写。


In [15]:
second_invoke_graph = build_timeline_graph(InMemorySaver())

print("第一次 invoke() 原始返回值：")
pprint(
    second_invoke_graph.invoke(
        {"seed": 10},
        {"configurable": {"thread_id": "second-invoke-timeline"}},
        durability="sync",
    ),
    sort_dicts=False,
)
print("\n第一次 invoke 后的 get_state()：")
pprint(
    second_invoke_graph.get_state(
        {"configurable": {"thread_id": "second-invoke-timeline"}}
    ),
    width=100,
)

checkpoint_ids_before = [
    snapshot.config["configurable"]["checkpoint_id"]
    for snapshot in reversed(
        list(
            second_invoke_graph.get_state_history(
                {
                    "configurable": {
                        "thread_id": "second-invoke-timeline"
                    }
                }
            )
        )
    )
]
print("\n第一次 invoke 的 checkpoint IDs（旧 → 新）：")
pprint(checkpoint_ids_before)

print("\n第二次 invoke() 原始返回值：")
pprint(
    second_invoke_graph.invoke(
        {"seed": 20},
        {"configurable": {"thread_id": "second-invoke-timeline"}},
        durability="sync",
    ),
    sort_dicts=False,
)
print("\n第二次 invoke 后的 get_state()：")
pprint(
    second_invoke_graph.get_state(
        {"configurable": {"thread_id": "second-invoke-timeline"}}
    ),
    width=100,
)

print("\n第二次 invoke 后的 checkpoint IDs（旧 → 新）：")
pprint([
    snapshot.config["configurable"]["checkpoint_id"]
    for snapshot in reversed(
        list(
            second_invoke_graph.get_state_history(
                {
                    "configurable": {
                        "thread_id": "second-invoke-timeline"
                    }
                }
            )
        )
    )
])
print(
    "旧 checkpoint IDs 仍是完整前缀：",
    [
        snapshot.config["configurable"]["checkpoint_id"]
        for snapshot in reversed(
            list(
                second_invoke_graph.get_state_history(
                    {
                        "configurable": {
                            "thread_id": "second-invoke-timeline"
                        }
                    }
                )
            )
        )
    ][: len(checkpoint_ids_before)] == checkpoint_ids_before,
)

print("\n第二次 invoke 后的完整 history（按创建顺序）：")
for snapshot in reversed(
    list(
        second_invoke_graph.get_state_history(
            {"configurable": {"thread_id": "second-invoke-timeline"}}
        )
    )
):
    pprint(snapshot, width=100)
    print()


第一次 invoke() 原始返回值：
{'seed': 10,
 'branch_results': ['left:11', 'right:12'],
 'joined': 'left:11 | right:12'}

第一次 invoke 后的 get_state()：
StateSnapshot(values={'seed': 10, 'branch_results': ['left:11', 'right:12'], 'joined': 'left:11 | right:12'}, next=(), config={'configurable': {'thread_id': 'second-invoke-timeline', 'checkpoint_ns': '', 'checkpoint_id': '1f19b1cb-e1d3-62e2-8002-8ab6193d2021'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-18T15:51:57.617016+00:00', parent_config={'configurable': {'thread_id': 'second-invoke-timeline', 'checkpoint_ns': '', 'checkpoint_id': '1f19b1cb-e1d1-6cee-8001-9d2bf6101d67'}}, tasks=(), interrupts=())

第一次 invoke 的 checkpoint IDs（旧 → 新）：
['1f19b1cb-e1cb-61f0-bfff-79daccd75702',
 '1f19b1cb-e1cd-6892-8000-9889ebb30774',
 '1f19b1cb-e1d1-6cee-8001-9d2bf6101d67',
 '1f19b1cb-e1d3-62e2-8002-8ab6193d2021']

第二次 invoke() 原始返回值：
{'seed': 20,
 'branch_results': ['left:11', 'right:12', 'left:21', 'right:22'],
 'joined': 'left:11

### 第二次 `invoke` 的精确含义

- **读取旧 checkpoint**：第二次调用按 `thread_id` 恢复上一轮最新状态；读取本身不会增加 history 数量。
- **写入新的 input checkpoint**：它有新 ID，`parent_config` 中的 checkpoint ID 指向旧最终 ID；其 `values` 是恢复的旧状态，`__start__` task 的 `result` 是本次输入。
- **合并本次输入**：下一 checkpoint 才把新输入放入 `values`，然后调度业务节点。
- **继续 superstep**：每个后续边界继续生成新 ID。旧的四个 checkpoint 仍是第二次历史的完整前缀。

所以“第二次调用开始时又看到旧状态”不等于重复保存旧的最新 snapshot。原始 history 会显示：旧 ID 仍是完整前缀，第一份新增 snapshot 有新 ID，并通过 `parent_config` 指回上一轮最终 ID。它是一份**新的输入边界 checkpoint**，承载本轮 `__start__` task。


## 8. 适用场景、限制与最佳实践

### `InMemorySaver` 适合

- 单元测试、Notebook、概念验证和本地调试；
- 在同一个 Python 进程中演示多轮调用、interrupt、state history 或 time travel；
- 不希望引入数据库前置条件的最小示例。

### 主要限制

- 进程退出或 saver 对象被丢弃后，checkpoint 消失；
- 多进程/多副本之间不会自动共享数据；
- 数据量随线程与历史增长占用进程内存，不适合作为生产持久化后端；
- 它不是跨线程长期记忆。checkpointer 按 `thread_id` 保存线程 State；若要让多个线程共享用户资料，应使用 Store。

### 最佳实践

1. 把 `thread_id` 当作持久化主键管理，保证租户隔离且避免复用错误；
2. 只把可序列化、真正需要恢复的数据放入 State，不把数据库连接、文件句柄等运行时资源塞入 checkpoint；
3. 用 `get_state()` 检查最新快照，用 `get_state_history()` 检查时间线；不要依赖内部字典结构；
4. 生产环境根据延迟与恢复目标选择 durability，并使用 PostgreSQL 等持久后端；
5. 图或 State Schema 升级时，要考虑旧 checkpoint 与新代码的兼容性。

### 官方资料

- [LangGraph Persistence](https://docs.langchain.com/oss/python/langgraph/persistence)
- [LangGraph Checkpointing API Reference](https://reference.langchain.com/python/langgraph/checkpoints)
